## Data Availability Notice

The data used in this notebook was provided by **Dogs Trust** under a formal data sharing agreement as part of an MSc Data Science dissertation at Liverpool John Moores University, supervised by Dr Jon Bielby.

**The raw data files are not included in this repository.** To reproduce the analysis, the following data extracts from the Dogs Trust Salesforce CRM are required:

- `Dog Affiliation and Status_Sub Status History Pull.xlsx` (Affiliation History sheet)
- `Sub Status History v2.csv`

Requests for data access should be directed to the Dogs Trust Data Science & Analytics team.

All data handling was conducted in accordance with GDPR and Dogs Trust's data governance protocols.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import openpyxl

# CSV files
status_history = pd.read_csv('../data/Status History.csv')
sub_status_history = pd.read_csv('../data/Sub status History.csv')

#Excel files
demographics = pd.read_excel('../data/MSc_extract_example - Dog Demographics.xlsx')
affiliation = pd.read_excel('../data/[Dogs Trust data file - not included in repo]')

In [ ]:
print(status_history.shape, sub_status_history.shape)
print(demographics.shape, affiliation.shape)

In [ ]:
print("Status History columns:", list(status_history.columns))
print("Sub Status History columns:", list(sub_status_history.columns))
print("Demographics columns:", list(demographics.columns))
print("Affiliation columns:", list(affiliation.columns))

In [ ]:
xl_file = pd.ExcelFile('../data/[Dogs Trust data file - not included in repo]')
print(xl_file.sheet_names)

In [ ]:
affiliation_history = pd.read_excel('../data/[Dogs Trust data file - not included in repo]', sheet_name='Affiliation History')

print(affiliation_history.shape)
print(affiliation_history.columns.tolist())


In [ ]:
print(affiliation_history['Process'].value_counts())

intake_only = affiliation_history[affiliation_history['Process'] == 'Intake']
intake_counts = intake_only.groupby('Dog ID').size()

print("\nDogs with multiple Intake records:")
print((intake_counts > 1).sum(), "out of", intake_counts.shape[0], "total dogs")
print("\nDistribution of intake counts:")
print(intake_counts.value_counts())

In [ ]:
sample_id = affiliation_history['Dog ID'].iloc[0]
dog_timeline = affiliation_history[affiliation_history['Dog ID'] == sample_id].sort_values('Start Date')
print(dog_timeline[['Process', 'Start Date', 'End Date', 'Account', 'Kennel Block/Location']])

In [ ]:
# Find a dog with a transfer record
transfer_dogs = affiliation_history[affiliation_history['Process'] == 'Transfer']['Dog ID'].unique()
sample_id = transfer_dogs[0]
dog_timeline = affiliation_history[affiliation_history['Dog ID'] == sample_id].sort_values('Start Date')
print(dog_timeline[['Process', 'Start Date', 'End Date', 'Account', 'Kennel Block/Location']])

In [ ]:
# Find a dog with a return record
return_dogs = affiliation_history[affiliation_history['Process'] == 'Return']['Dog ID'].unique()
sample_id = return_dogs[0]
dog_timeline = affiliation_history[affiliation_history['Dog ID'] == sample_id].sort_values('Start Date')
print(dog_timeline[['Process', 'Start Date', 'End Date', 'Account', 'Kennel Block/Location']])

In [ ]:
demographics_ids = set(demographics['dog_id'])
affiliation_ids = set(affiliation_history['Dog ID'])

print("Demographics unique dogs:", len(demographics_ids))
print("Affiliation unique dogs:", len(affiliation_ids))
print("Dogs in both:", len(demographics_ids & affiliation_ids))
print("Dogs only in demographics:", len(demographics_ids - affiliation_ids))
print("Dogs only in affiliation:", len(affiliation_ids - demographics_ids))

In [ ]:
print(affiliation_history[['Sex', 'Date of Birth', 'Dog Breed', 'Size', 'Dog Handling Grade']].notna().sum())

In [ ]:
removed_group = pd.read_excel('../data/[Dogs Trust data file - not included in repo]', sheet_name='REMOVED Group')

print(removed_group.shape)
print(removed_group.columns.tolist())
removed_group.head()

In [ ]:
print(removed_group['Removed Group'].value_counts())

In [ ]:
# Sort by dog and date to get correct sequence
sorted_aff = affiliation_history.sort_values(['Dog ID', 'Start Date'])

# Get the next process for each row within each dog
sorted_aff['Next Process'] = sorted_aff.groupby('Dog ID')['Process'].shift(-1)

# Drop rows where there's no next process
transitions = sorted_aff.dropna(subset=['Next Process'])

# Count transitions
transition_counts = transitions.groupby(['Process', 'Next Process']).size().reset_index(name='Count')
print(transition_counts)

In [ ]:
# Manually order nodes left to right
node_order = ['Intake', 'Movement', 'Transfer', 'Adoption', 'Return']
process_to_idx = {p: i for i, p in enumerate(node_order)}

# x positions spread left to right
x_positions = [0.01, 0.3, 0.3, 0.6, 0.9]

fig = go.Figure(data=[go.Sankey(
    arrangement='snap',
    node=dict(
        pad=20,
        thickness=20,
        line=dict(color="black", width=0.5),
        label=node_order,
        x=x_positions,
        y=[0.5]*len(node_order)
    ),
    link=dict(
        source=transition_counts['Process'].map(process_to_idx),
        target=transition_counts['Next Process'].map(process_to_idx),
        value=transition_counts['Count']
    )
)])

fig.update_layout(title_text="Dog Journey: Process Transitions", font_size=12)
fig.show()

In [ ]:
fig.write_image('../outputs/dog_journey_process.png', width=1400, height=500, scale=2)
fig.write_html('../outputs/dog_journey_process.html') # Interactive version

In [ ]:
status_excel = pd.read_excel('../data/[Dogs Trust data file - not included in repo]', sheet_name='Status History')

print("CSV shape:", status_history.shape)
print("Excel shape:", status_excel.shape)
print("Columns match:", list(status_history.columns) == list(status_excel.columns))

In [ ]:
print("CSV columns:", list(status_history.columns))
print("Excel columns:", list(status_excel.columns))
print("CSV field values:", status_history['field'].unique())
print("Excel field values:", status_excel[status_excel.columns[1]].unique())  # adjust column name once we see it

In [ ]:
print("CSV date range:", status_history['edit_date'].min(), "to", status_history['edit_date'].max())
print("Excel date range:", status_excel['Edit Date'].min(), "to", status_excel['Edit Date'].max())
print("CSV unique dogs:", status_history['dog_id'].nunique())
print("Excel unique dogs:", status_excel['Dog ID'].nunique())

In [ ]:
sub_status_excel = pd.read_excel('../data/[Dogs Trust data file - not included in repo]', sheet_name='Sub status History')

print("CSV shape:", sub_status_history.shape)
print("Excel shape:", sub_status_excel.shape)
print("Columns match:", list(sub_status_history.columns) == list(sub_status_excel.columns))

In [ ]:
print("CSV date range:", sub_status_history['edit_date'].min(), "to", sub_status_history['edit_date'].max())
print("Excel date range:", sub_status_excel['Edit Date'].min(), "to", sub_status_excel['Edit Date'].max())

print("CSV unique dogs:", sub_status_history['dog_id'].nunique())
print("Excel unique dogs:", sub_status_excel['Dog ID'].nunique())

In [ ]:
# Count each Process type
process_counts = affiliation_history['Process'].value_counts().reset_index()
process_counts.columns = ['Process', 'Count']

# Build sunburst data
# We need: id, parent, value, label
ids = ['Dogs Trust'] + list(process_counts['Process'])
labels = ['Dogs Trust'] + list(process_counts['Process'])
parents = [''] + ['Dogs Trust'] * len(process_counts)
values = [0] + list(process_counts['Count'])

fig = px.sunburst(
    names=labels,
    parents=parents,
    values=values,
    title='Dog Trust Rehoming Process - Affiliation History Breakdown'
)

fig.update_traces(textinfo='label+percent entry')
fig.update_layout(margin=dict(t=50, l=0, r=0, b=0))
fig.show()

In [ ]:
fig.write_image('../outputs/dog_trust_affiliation_breakdown.png', width=1200, height=800, scale=2)
fig.write_html('../outputs/dog_trust_affiliation_breakdown.html') # Interactive version

In [ ]:
# Build a Multi-ring sunburst chart to show the breakdown of Processes and Sub Processes

movement_record = affiliation_history[affiliation_history['Process'] == 'Movement'].copy()

def classify_location(val):
    if pd.isna(val) or val == '':
        return 'No Location Recorded'
    elif 'Home From Home' in str(val):
        return 'Home From Home'
    elif 'vehicle' in str(val).lower() or 'transfer' in str(val).lower():
        return 'Transfer Vehicle'
    else:
        return 'Physical Kennel'
    
movement_record['Location Type'] = movement_record['Kennel Block/Location'].apply(classify_location)    
print(movement_record['Location Type'].value_counts())

In [ ]:
# Define the hierarchy: Process - sub-category

# Adoption breakdown
adoption_active = affiliation_history[
    (affiliation_history['Process'] == 'Adoption') & 
    (affiliation_history['End Date'].isna())
].shape[0]

adoption_closed = affiliation_history[
    (affiliation_history['Process'] == 'Adoption') & 
    (affiliation_history['End Date'].notna())
].shape[0]

# Intake breakdown
intake_counts = affiliation_history[affiliation_history['Process'] == 'Intake'].groupby('Dog ID').size()
single_intake = (intake_counts == 1).sum()
multiple_intake = (intake_counts > 1).sum()

print(adoption_active, adoption_closed, single_intake, multiple_intake)

In [ ]:
# Get the sequence of processes per dog
dog_sequences = affiliation_history.sort_values(['Dog ID', 'Start Date'])\
    .groupby('Dog ID')['Process'].apply(list).reset_index()
dog_sequences.columns = ['Dog ID', 'Journey']

# Classify each dog's journey
def classify_journey(processes):
    if 'Return' in processes:
        return 'Return Journey'
    elif 'Transfer' in processes:
        return 'Transfer Journey'
    else:
        return 'Simple Journey'

dog_sequences['Journey Type'] = dog_sequences['Journey'].apply(classify_journey)
print(dog_sequences['Journey Type'].value_counts())

In [ ]:
# Add journey type back to affiliation history
affiliation_with_journey = affiliation_history.merge(
    dog_sequences[['Dog ID', 'Journey Type']], 
    on='Dog ID', 
    how='left'
)

# Count Process occurrences within each journey type
journey_process_counts = affiliation_with_journey.groupby(
    ['Journey Type', 'Process']
).size().reset_index(name='Count')

print(journey_process_counts)

In [ ]:
# Count distinct dogs per journey type per process
journey_dog_counts = affiliation_with_journey.groupby(
    ['Journey Type', 'Process']
)['Dog ID'].nunique().reset_index(name='Dog Count')

print(journey_dog_counts)

In [ ]:
# Journey archetype distribution - Ring 1 only
# Outer ring removed due to complexity. Precise process counts per archetype are in journey_dog_counts above.
labels = [
    'Dogs Trust',
    'Simple Journey', 'Transfer Journey', 'Return Journey',
]

parents = [
    '',
    'Dogs Trust', 'Dogs Trust', 'Dogs Trust',
]

values = [
    0,
    10476, 2885, 1475,
 ]

In [ ]:
fig_sunburst = px.sunburst(
    names=labels,
    parents=parents,
    values=values,
    title='Dogs Trust Rehoming Process — Journey Archetype Breakdown',
    color_discrete_sequence=px.colors.qualitative.Pastel
)

fig_sunburst.update_traces(textinfo='label+percent parent', textfont_size=16)
fig_sunburst.update_layout(
    margin=dict(t=80, l=80, r=80, b=80),
    height=800,
    width=1000
)
fig_sunburst.show()

In [ ]:
# Save Journey Archetype Sunburst
fig_sunburst.write_image('../outputs/dog_trust_process_sunburst.png', width=1200, height=800, scale=2)
fig_sunburst.write_html('../outputs/dog_trust_process_sunburst.html') # Interactive version

In [ ]:
# Get Intake start date per dog (first Intake record)
intake_dates = affiliation_history[affiliation_history['Process'] == 'Intake']\
    .sort_values('Start Date')\
    .groupby('Dog ID')['Start Date'].first().reset_index()
intake_dates.columns = ['Dog ID', 'Intake Date']

# Get last record per dog
last_records = affiliation_history.sort_values('Start Date')\
    .groupby('Dog ID').last().reset_index()

# Use End Date if available, otherwise fall back to Start Date
last_records['Exit Date'] = last_records['End Date'].fillna(last_records['Start Date'])

# Merge and calculate length of stay
los_df = intake_dates.merge(last_records[['Dog ID', 'Exit Date']], on='Dog ID')
los_df['Length of Stay'] = (
    pd.to_datetime(los_df['Exit Date']) - pd.to_datetime(los_df['Intake Date'])
).dt.days

print(los_df['Length of Stay'].describe())

In [ ]:
# Define the length of stay bands according to Dogs trust operational definitions
# 0-60: normal adoption range, 60-182: medium-term, 183+: long term stay

bins = [0, 60, 182, float('inf')]
labels = ['0-60 days (Normal)', '60-182 days (Medium)', '183+ days (Long-term)']

los_df['LOS Band'] = pd.cut(los_df['Length of Stay'], bins=bins, labels=labels, right=True)

print(los_df['LOS Band'].value_counts().sort_index())
print(f"\nTotal dogs: {len(los_df)}")


In [ ]:
# Merge LOS band back to affiliation history for further analysis
affiliation_with_los = affiliation_history.merge(
    los_df[['Dog ID', 'LOS Band']], on='Dog ID', how='left')

print(affiliation_with_los['LOS Band'].value_counts().sort_index())

In [ ]:
bands = ['0-60 days (Normal)', '60-182 days (Medium)', '183+ days (Long-term)']
node_order = ['Intake', 'Movement', 'Transfer', 'Adoption', 'Return']
process_to_idx = {p: i for i, p in enumerate(node_order)}
x_positions = [0.01, 0.3, 0.3, 0.6, 0.9]

for band in bands:
    band_df = affiliation_with_los[affiliation_with_los['LOS Band'] == band].copy()

    # Calculate transitions
    sorted_band = band_df.sort_values(['Dog ID', 'Start Date'])
    sorted_band['Next Process'] = sorted_band.groupby('Dog ID')['Process'].shift(-1)
    transitions_band = sorted_band.dropna(subset=['Next Process'])
    transistion_counts = transitions_band.groupby(['Process', 'Next Process']).size().reset_index(name='Count')

    # Filter to known processes only
    transistion_counts = transistion_counts[
    transistion_counts['Process'].isin(node_order) &
    transistion_counts['Next Process'].isin(node_order)
    ]

    fig_LOS = go.Figure(data=[go.Sankey(
        arrangement='snap',
        node=dict(
            pad=20,
            thickness=20,
            line=dict(color="black", width=0.5),
            label=node_order,
            x=x_positions,
            y=[0.5]*len(node_order)
        ),
        link=dict(
            source=transistion_counts['Process'].map(process_to_idx),
            target=transistion_counts['Next Process'].map(process_to_idx),
            value=transistion_counts['Count']
        )
    )])

    fig_LOS.update_layout(
        title_text=f"Dog Journey: Process Transitions for {band}",
        font_size=12,
        height=400)
    
    # Save plots
    safe_band = band.replace("+", "plus").replace("-", "_").replace(" ", "_")
    fig_LOS.write_image(f'../outputs/dog_journey_process_LOS_{safe_band}.png', width=1400, height=500, scale=2)
    fig_LOS.write_html(f'../outputs/dog_journey_process_LOS_{safe_band}.html')
    fig_LOS.show()
    

In [ ]:
# Calculate the average number of Movement records per dog for each LOS band
movement_frequency = affiliation_with_los[
    affiliation_with_los['Process'] == 'Movement'
].groupby(['LOS Band', 'Dog ID']).size().reset_index(name='Movement Count')

avg_movements = movement_frequency.groupby('LOS Band')['Movement Count'].mean().reset_index()
avg_movements.columns = ['LOS Band', 'Avg Movements per Dog']

# Calculate adoption rate per LOS band
adoption_counts = affiliation_with_los[
    affiliation_with_los['Process'] == 'Adoption'
].groupby('LOS Band')['Dog ID'].nunique().reset_index(name=' Dogs Adopted')

dog_counts = los_df.groupby('LOS Band').size().reset_index(name='Total Dogs')

adoption_rate = dog_counts.merge(adoption_counts, on='LOS Band', how='left')
adoption_rate['Adoption Rate'] = (adoption_rate[' Dogs Adopted'] / adoption_rate['Total Dogs'] * 100).round(1)

print("Average Movements per Dog by LOS Band:")
print(avg_movements.sort_values('LOS Band'))
print("\nAdoption Rate by LOS Band:")
print(adoption_rate.sort_values('LOS Band'))

In [ ]:
# Total kennel days consumed per band
kennel_days = los_df.groupby('LOS Band')['Length of Stay'].agg(
    ['count', 'mean', 'sum']
).reset_index()

kennel_days.columns = ['LOS Band', 'Total Dogs', 'Avg LOS', 'Total Kennel Days']
kennel_days['Avg LOS'] = kennel_days['Avg LOS'].round(1)
kennel_days['Total Kennel Days'] = kennel_days['Total Kennel Days'].round(0).astype(int)
kennel_days['Pct of Total'] = (
    kennel_days['Total Kennel Days'] / kennel_days['Total Kennel Days'].sum() * 100
).round(1)

print(kennel_days.sort_values('LOS Band').to_string(index=False))

In [ ]:
fig_kennel = px.bar(
    kennel_days.sort_values('LOS Band'),
    x='LOS Band',
    y='Total Kennel Days',
    title='Total Kennel Days Consumed by Length of Stay Band',
    labels={'Total Kennel Days': 'Total Kennel Days', 'LOS Band': 'Length of Stay Band'},
    color='LOS Band',
    color_discrete_sequence=px.colors.qualitative.Pastel,
    custom_data=['Pct of Total']
)

fig_kennel.update_traces(
    texttemplate='%{y:,.0f}<br>(%{customdata[0]}%)',
    textposition='outside'
)

fig_kennel.update_layout(
    showlegend=False,
    height=500,
    width=800,
    yaxis_range=[0, 390000]
)

fig_kennel.show()

In [ ]:
fig_kennel.write_image('../outputs/kennel_days_by_los_band.png', width=1000, height=600, scale=2)
fig_kennel.write_html('../outputs/kennel_days_by_los_band.html') # Interactive version